# U02_C1 — Entrez Search from Python
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 2:** Information Retrieval from Biological Databases
**Type:** Course notebook · **Time:** 40 min · **Difficulty:** ★★☆
**Goes with:** Lectures L3–L4, Practicals S2–S3 (NCBI & PubMed Retrieval)
**Before this:** U01_C1

## 1. What you will learn
- **Explain** the Entrez integrated retrieval system and RefSeq accession prefixes [LO1, LO2].
- **Construct** precise queries using field tags (`[GENE]`, `[ORGN]`, `[FILT]`) and Boolean operators [LO2].
- **Search** and retrieve biological records programmatically with `Bio.Entrez` (`esearch`, `esummary`, `efetch`) [LO2].
- **Implement** an offline-first fallback mechanism for robust classroom execution [LO2].


## 2. Biological question
A clinical geneticist identifies a mutation suspected of causing beta-thalassemia. To interpret the mutation, they need the official curated reference transcript for human hemoglobin subunit beta (*HBB*). If you simply type `HBB` into the NCBI search bar, you get over 50,000 nucleotide records! How do we write a search query that returns *only* the official, curated human *HBB* RefSeq mRNA (`NM_000518.5`), and how do we fetch this record directly into Python?


In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U02_C1"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

## 3. Concept 1: The Entrez System and RefSeq Accession Prefixes
### Theory
NCBI **Entrez** is an integrated database cross-referencing system connecting literature (PubMed), genes (Gene), sequences (Nucleotide, Protein), structures (Structure), and taxonomy (Taxonomy).

To distinguish raw submitted data from curated benchmarks, NCBI maintains **RefSeq** (Reference Sequence Database). RefSeq accessions contain an underscore `_` and standard two-letter prefixes:
- `NM_`: Curated mRNA transcript (e.g. `NM_000518` for human *HBB*)
- `NP_`: Curated protein sequence (e.g. `NP_000509` for beta-globin)
- `NR_`: Curated non-coding RNA
- `NC_`: Complete genomic molecule / chromosome (e.g. viral genome or human chromosome)
- `XM_` / `XP_`: Model (computationally predicted) mRNA and protein

Primary GenBank accessions (e.g. `V00497`, `AF123456`) do *not* have underscores.


In [ ]:
# Worked Example: Distinguishing accession types in Python
accessions = ["NM_000518.5", "NP_000509.1", "NC_001422.1", "V00497.1", "P68871"]

def classify_accession(acc):
    if "_" in acc:
        prefix = acc.split("_")[0]
        mapping = {
            "NM": "RefSeq Curated mRNA",
            "NP": "RefSeq Curated Protein",
            "NC": "RefSeq Complete Chromosome / Genome",
            "XM": "RefSeq Predicted mRNA",
            "XP": "RefSeq Predicted Protein"
        }
        return mapping.get(prefix, f"Other RefSeq ({prefix})")
    elif len(acc) == 6 and acc[0] in "OPQ":
        return "UniProtKB Accession"
    else:
        return "Primary GenBank / INSDC Submission"

for a in accessions:
    print(f"{a:15} -> {classify_accession(a)}")


## 4. Concept 2: Query Syntax with Field Tags
### Theory
Searching without field tags searches all fields (text words, titles, authors, organisms). **Field tags** restrict search terms to specific indexed fields:
- `[GENE]` or `[Gene Name]`: Official gene symbol (e.g., `HBB[GENE]`)
- `[ORGN]` or `[Organism]`: Scientific or common name (e.g., `Homo sapiens[ORGN]`)
- `[PROP]` or `[FILT]`: Property or filter flag (e.g., `biomol_mrna[PROP]`, `RefSeq[FILT]`)
- `[TITL]`: Article or sequence definition title words

Boolean operators must be **UPPERCASE** (`AND`, `OR`, `NOT`).


In [ ]:
# Worked Example: Precise Entrez Query String
query_broad = "HBB"
query_precise = "HBB[GENE] AND Homo sapiens[ORGN] AND biomol_mrna[PROP] AND RefSeq[FILT]"

print("Broad search query:  ", query_broad)
print("Precise search query:", query_precise)


## 5. Concept 3: Programmatic Search with Bio.Entrez (esearch)
### Theory
Biopython provides the `Bio.Entrez` module to access NCBI E-utilities:
1. `Entrez.email`: Must be set to identify yourself to NCBI (usage policy).
2. `Entrez.esearch(db, term, ...)`: Performs the search and returns XML/JSON listing matching Primary IDs (UIDs or GIs).
3. `Entrez.read(handle)`: Parses NCBI XML responses into native Python dictionaries.

**Offline-First Fallback:** Whenever calling NCBI servers in teaching code, wrap the network call in a `try...except` block so that network interruptions fallback to pre-saved records in `data/expected/`.


In [ ]:
# Worked Example: Searching Nucleotide via Bio.Entrez with offline fallback
import json
from Bio import Entrez

Entrez.email = "student@example.edu"  # Use your own educational email in practice

query = "HBB[GENE] AND Homo sapiens[ORGN] AND biomol_mrna[PROP] AND RefSeq[FILT]"
offline_file = DATA_DIR / "expected" / "entrez_hbb_search.json"

try:
    # Attempt real Entrez search
    handle = Entrez.esearch(db="nucleotide", term=query, retmax=5)
    record = Entrez.read(handle)
    handle.close()
    count = int(record["Count"])
    id_list = record["IdList"]
    print("NCBI ESearch succeeded online!")
except Exception as e:
    # Offline fallback
    print(f"Network call skipped or unavailable ({e}). Loading offline fallback...")
    with open(offline_file) as f:
        data = json.load(f)
    count = int(data["count"])
    id_list = data["idlist"]

print(f"Query: {query}")
print(f"Matching records found: {count}")
print(f"Retrieved primary UIDs: {id_list}")


In [ ]:
# Try it: Change the query term to search for human insulin (INS)
ins_query = "INS[GENE] AND Homo sapiens[ORGN] AND biomol_mrna[PROP] AND RefSeq[FILT]"
print("Constructed query:", ins_query)


**Common mistakes:**
- Forgetting `Entrez.email`: NCBI will throttle or temporarily block IP addresses that fail to supply an email header.
- Lowercase boolean operators: `"hbb and human"` treats `and` as a literal word search rather than a boolean operator. Always write `AND`, `OR`, `NOT`.


## 6. Concept 4: Fetching Complete Records with Bio.Entrez (efetch)
### Theory
`Entrez.efetch` downloads complete records in standard file formats:
- `db`: target database (`nucleotide`, `protein`, `pubmed`)
- `id`: accessions or UIDs
- `rettype`: return format (`gb` or `gbwithparts` for GenBank, `fasta` for FASTA, `medline` for PubMed)
- `retmode`: text or xml

The resulting handle can be parsed directly by `Bio.SeqIO.read()`!


In [ ]:
# Worked Example: efetch parsed directly into SeqRecord
from Bio import SeqIO

accession = "NM_000518.5"
local_gb = DATA_DIR / "genbank" / "hbb_human.gb"

try:
    handle = Entrez.efetch(db="nucleotide", id=accession, rettype="gbwithparts", retmode="text")
    seq_rec = SeqIO.read(handle, "genbank")
    handle.close()
    print(f"Retrieved {accession} online via efetch!")
except Exception as e:
    print(f"Using local cached copy ({e})...")
    seq_rec = SeqIO.read(local_gb, "genbank")

print("Record ID:         ", seq_rec.id)
print("Record Description:", seq_rec.description)
print("Molecule Type:     ", seq_rec.annotations.get("molecule_type", "mRNA"))
print("Sequence Length:   ", len(seq_rec.seq), "bp")


## 7. Compare with the Web Tool
In Practical S2, we ran the same search in the NCBI web search bar:
`HBB[GENE] AND Homo sapiens[ORGN] AND biomol_mrna[PROP] AND RefSeq[FILT]`
The web interface returns the official RefSeq record `NM_000518.5` (Homo sapiens hemoglobin subunit beta mRNA, 626 bp). Our Python `Bio.Entrez` script retrieves exactly the same entry and parses its features.


## 8. Summary
- RefSeq accessions contain underscores (`NM_` for mRNA, `NP_` for protein, `NC_` for complete genomes).
- Entrez field tags (`[GENE]`, `[ORGN]`, `[FILT]`) narrow search space from tens of thousands of hits to exact curated records.
- `Bio.Entrez.esearch` searches for IDs; `Bio.Entrez.efetch` downloads complete formatted records.
- Always implement an offline fallback using `try...except` and local files.

| Function | Purpose | Input Arguments | Output |
|---|---|---|---|
| `Entrez.esearch` | Search database | `db`, `term`, `retmax` | XML handle containing `IdList`, `Count` |
| `Entrez.esummary` | Document summary | `db`, `id` | XML handle containing title, date, length |
| `Entrez.efetch` | Download records | `db`, `id`, `rettype`, `retmode` | Text/XML handle parsed by `SeqIO` |
| `Entrez.read` | Parse NCBI XML | handle | Python dict / list |


## 9. Next Steps
- Practice building queries and handling responses in `U02_P_Entrez_Search.ipynb`.
- Test your query logic and RefSeq prefix knowledge in `U02_Q_Information_Retrieval.ipynb`.
